# R/S benchmark — 2. PCE training

Stage 2 of the split benchmark pipeline. This notebook **only** fits and validates the PCE — it
makes no emulator calls and draws no latent samples. It reads the `dataset_unique_train` /
`dataset_unique_val` files written by
[`01_generate_dataset.ipynb`](01_generate_dataset.ipynb).

The random-variable definitions below (`r_mean`, `r_std`, ...) must match stage 1's — they are only
used to rebuild the `joint` distribution for the PCE's polynomial basis, not to draw new samples.

Functions come from [`functions_final.py`](../functions_final.py):
`train_and_validate_pce_from_dataset_benchmark` fits one PCE per time step and scores it, writing
the same `pce_metamodel` / `pce_validation_stats` artefacts the original combined pipeline did.

## 1. Libraries

In [ ]:
import sys
import time
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

import dill
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

from functions_final import *
from UQpy.distributions import Normal, JointIndependent

## 2. Random variables and fixed parameters

Must match [`01_generate_dataset.ipynb`](01_generate_dataset.ipynb) — this only rebuilds the
distribution object, it draws no new samples.

In [ ]:
r_mean = 5.0
r_std  = 0.8
s_mean = 2.0
s_std  = 0.6

n_latent_samples = 100000   # must match stage 1 — it is the filename prefix
n_lambdas        = 4
max_degree       = 3        # maximum total degree of the PCE polynomial basis

r_dist = Normal(loc=r_mean, scale=r_std)
s_dist = Normal(loc=s_mean, scale=s_std)
joint  = JointIndependent(marginals=[r_dist, s_dist])

## 3. Time grid

Must match stage 1 — these values name the files being loaded.

In [ ]:
times = np.linspace(0, 150, 10, endpoint=True)
times

## 4. Load the datasets and train the PCE at each time step

No emulator calls below — `train_and_validate_pce_from_dataset_benchmark` only fits a
`PolynomialChaosExpansion` on the lambdas already computed by stage 1 and scores it against the
saved validation split.

In [ ]:
print("="*60)
print("TRAINING THE BENCHMARK PCE")
print("="*60)

results = []
for t in times:
    with open(f'{n_latent_samples}_dataset_unique_train_{t}_benchmark.pkl', 'rb') as f:
        df_unique_train = dill.load(f)
    with open(f'{n_latent_samples}_dataset_unique_val_{t}_benchmark.pkl', 'rb') as f:
        df_unique_val = dill.load(f)

    result = train_and_validate_pce_from_dataset_benchmark(
                                                              df_unique_train=df_unique_train,
                                                              df_unique_val=df_unique_val,
                                                              joint=joint,
                                                              time_step=t,
                                                              n_latent_samples=n_latent_samples,
                                                              n_lambdas=n_lambdas,
                                                              max_degree=max_degree,
                                                              output_dir='.',
                                                          )
    result['x_train'] = df_unique_train[['r', 's']].to_numpy()
    results.append(result)

## 5. Validation summary

How well the PCE reproduces each lambda, per time step.

In [ ]:
validation_summary = pd.concat([r['statistics'] for r in results], ignore_index=True)
validation_summary.insert(0, 'Time (years)', [r['time_step'] for r in results])
validation_summary

## 6. Emulator efficiency (speed-up)

Combines this notebook's PCE evaluation time with the emulator cost recorded by stage 1
(`<n_latent_samples>_emulator_timing_benchmark.pkl`) to get the speed-up: how much cheaper it is to
evaluate the surrogate than to rebuild the dataset it was trained on.

In [ ]:
with open(f'{n_latent_samples}_emulator_timing_benchmark.pkl', 'rb') as f:
    emulator_timing = dill.load(f)

speedup_rows = []
for result in results:
    emulator_s = float(emulator_timing.loc[emulator_timing['Time (years)'] == result['time_step'], 'Train total (s)'].iloc[0])

    t_start = time.perf_counter()
    result['pce_metamodel'].predict(result['x_train'])
    surrogate_s = time.perf_counter() - t_start

    speedup_rows.append({
                            'Time (years)':  result['time_step'],
                            'Emulator (s)':  emulator_s,
                            'Surrogate (s)': surrogate_s,
                            'Speed-up':      emulator_s / surrogate_s,
                        })

speedup = pd.DataFrame(speedup_rows)
print(f"Median speed-up: {speedup['Speed-up'].median():,.0f}x")
speedup

### 6.1 Speed-up over time

In [ ]:
fig, ax = plt.subplots(figsize=(6, 3.5))
ax.plot(speedup['Time (years)'], speedup['Speed-up'], marker='o', color='0.25')
ax.set_xlabel('Time (years)')
ax.set_ylabel('Speed-up (emulator / surrogate)')
ax.set_yscale('log')
ax.yaxis.set_major_formatter(ticker.FuncFormatter(lambda v, _: f'{v:,.0f}x'))
ax.grid(True, which='both', alpha=0.3)
plt.tight_layout()
plt.show()